In [ ]:
# ============================================================
#  PROJECT 3: AI Recommendation Logic
#  Tech Stack Recommender | DecodeLabs | Batch 2026
#  Algorithm: TF-IDF Weighting + Cosine Similarity
# ============================================================

import math


# ── Dataset ──────────────────────────────────────────────────
# Skills listed multiple times = more important for that role.
# Repetition drives TF (Term Frequency) weighting.

job_roles = {
    "Data Scientist": [
        "python", "python", "machine learning", "machine learning",
        "sql", "statistics", "statistics", "data analysis",
        "tensorflow", "pandas", "numpy", "data analysis"
    ],
    "Web Developer": [
        "html", "html", "css", "css", "javascript", "javascript",
        "react", "nodejs", "sql", "git", "apis", "apis"
    ],
    "DevOps Engineer": [
        "aws", "aws", "docker", "docker", "kubernetes", "kubernetes",
        "linux", "git", "automation", "automation", "ci/cd", "cloud"
    ],
    "Mobile App Developer": [
        "java", "java", "kotlin", "swift", "react",
        "apis", "git", "firebase", "firebase", "android", "android"
    ],
    "Cybersecurity Analyst": [
        "networking", "networking", "linux", "python",
        "encryption", "encryption", "ethical hacking", "ethical hacking",
        "firewalls", "sql", "bash", "bash"
    ],
    "Cloud Architect": [
        "aws", "aws", "azure", "azure", "cloud", "cloud",
        "docker", "kubernetes", "automation", "networking", "linux", "linux"
    ],
    "AI/ML Engineer": [
        "python", "python", "tensorflow", "tensorflow",
        "machine learning", "machine learning", "deep learning", "deep learning",
        "numpy", "pandas", "statistics", "data analysis"
    ],
    "Database Administrator": [
        "sql", "sql", "mysql", "mysql", "postgresql", "mongodb",
        "data analysis", "linux", "backup", "backup",
        "performance tuning", "performance tuning"
    ],
    "Backend Developer": [
        "python", "java", "java", "sql", "apis", "apis",
        "nodejs", "nodejs", "git", "docker", "linux", "linux"
    ],
    "UI/UX Designer": [
        "figma", "figma", "html", "css", "javascript",
        "user research", "user research", "prototyping", "prototyping",
        "wireframing", "wireframing", "react"
    ],
}


# ── Vocabulary ───────────────────────────────────────────────

def build_vocabulary(job_data):
    """Return a sorted list of all unique skills across every role."""
    all_skills = set()
    for skills in job_data.values():
        all_skills.update(skills)
    return sorted(all_skills)


# ── TF-IDF Vectorizer ────────────────────────────────────────

def calculate_tfidf(skill_list, vocabulary, all_role_skills):
    """
    Convert a skill list into a TF-IDF weighted vector.

    TF  = how often a skill appears in this role / total skills in role
    IDF = log(total roles / roles that contain this skill)
          Rare skills get higher weight. Common skills get lower weight.
    """
    total_terms = len(skill_list)
    total_docs  = len(all_role_skills)
    vector      = []

    for skill in vocabulary:
        count = skill_list.count(skill)

        if count == 0:
            vector.append(0.0)
            continue

        tf = count / total_terms

        # Count how many roles contain this skill
        docs_with_skill = 0
        for role_skills in all_role_skills:
            if skill in role_skills:
                docs_with_skill += 1

        idf = math.log(total_docs / docs_with_skill)

        vector.append(tf * idf)

    return vector


# ── Cosine Similarity ────────────────────────────────────────

def cosine_similarity(vec_a, vec_b):
    """
    Measure the angle between two vectors.
    Returns a score from 0 (no match) to 1 (perfect match).

    cos(θ) = (A · B) / (||A|| × ||B||)
    """
    # Dot product
    dot_product = 0
    for i in range(len(vec_a)):
        dot_product += vec_a[i] * vec_b[i]

    # Magnitudes
    mag_a = math.sqrt(sum(x ** 2 for x in vec_a))
    mag_b = math.sqrt(sum(x ** 2 for x in vec_b))

    # Avoid division by zero (cold start situation)
    if mag_a == 0 or mag_b == 0:
        return 0.0

    return dot_product / (mag_a * mag_b)


# ── Recommendation Pipeline ───────────────────────────────────

def get_recommendations(user_skills, job_data, vocabulary, top_n=3):
    """
    Run the 4-step pipeline: Ingestion → Scoring → Sorting → Filtering.
    Returns the top_n most similar roles with their scores and matched skills.
    """
    all_role_skills = list(job_data.values())

    # Step 1 — Ingestion: build user TF-IDF vector
    user_vector = calculate_tfidf(user_skills, vocabulary, all_role_skills)

    # Step 2 — Scoring: compare against every job role
    results = []
    for role, role_skills in job_data.items():
        role_vector = calculate_tfidf(role_skills, vocabulary, all_role_skills)
        score       = cosine_similarity(user_vector, role_vector)
        matched     = [s for s in user_skills if s in role_skills]
        results.append((score, role, matched))

    # Step 3 — Sorting: highest score first
    results.sort(reverse=True)

    # Step 4 — Filtering: return only top N results
    return results[:top_n]


# ── Cold Start Check ──────────────────────────────────────────

def check_recognized_skills(user_skills, vocabulary):
    """Split user input into recognized and unrecognized skills."""
    recognized   = [s for s in user_skills if s in vocabulary]
    unrecognized = [s for s in user_skills if s not in vocabulary]
    return recognized, unrecognized


# ── UI Helpers ────────────────────────────────────────────────

def display_welcome():
    print("\n" + "=" * 58)
    print("   🤖  TECH STACK RECOMMENDER — DecodeLabs AI")
    print("   Project 3: AI Recommendation Logic | Batch 2026")
    print("=" * 58)
    print("   Enter your skills → get matched to the best job roles\n")


def get_user_skills():
    """Collect at least 3 unique skills from the user."""
    print("📌 Example skills you can enter:")
    print("   python, sql, java, aws, docker, javascript,")
    print("   machine learning, html, css, linux, react,")
    print("   tensorflow, git, cloud, deep learning, figma,")
    print("   kotlin, data analysis, bash, networking...\n")
    print("   Type each skill and press Enter.")
    print("   Type 'done' when finished (minimum 3 skills).\n")

    user_skills = []

    while True:
        entry = input(f"   Skill #{len(user_skills) + 1}: ").strip().lower()

        if entry == "done":
            if len(user_skills) < 3:
                print("   ⚠️  Please enter at least 3 skills.\n")
            else:
                break
        elif entry == "":
            print("   ⚠️  Cannot be empty. Try again.")
        elif entry in user_skills:
            print(f"   ⚠️  '{entry}' is already added. Try a different skill.")
        else:
            user_skills.append(entry)
            print(f"   ✅ Added: {entry}")

    return user_skills


def display_results(results, user_skills, unrecognized):
    print("\n" + "=" * 58)
    print("   🎯  YOUR TOP JOB ROLE RECOMMENDATIONS")
    print("=" * 58)
    print(f"\n   Skills entered : {', '.join(user_skills)}")

    # Warn user about any skills not in our dataset (cold start)
    if unrecognized:
        print(f"   ⚠️  Not in dataset : {', '.join(unrecognized)} (skipped)")

    print()

    medals    = ["🥇", "🥈", "🥉"]
    any_match = False

    for rank, (score, role, matched) in enumerate(results):

        # Skip results with no meaningful similarity
        if score < 0.05:
            continue

        any_match   = True
        pct         = round(score * 100, 2)
        filled      = int(pct / 5)
        bar         = "█" * filled + "░" * (20 - filled)
        medal       = medals[rank] if rank < 3 else "🔹"
        matched_str = ", ".join(matched) if matched else "none directly"

        print(f"   {medal}  Rank {rank + 1}: {role}")
        print(f"         Score   : {pct}%  [{bar}]")
        print(f"         Matched : {matched_str}\n")

    # Cold start fallback message
    if not any_match:
        print("   ❌ No strong matches found (Cold Start Problem).")
        print("   💡 Your skills may not be in our dataset yet.")
        print("      Try skills from the examples list above.\n")

    print("=" * 58)


# ── Entry Point ───────────────────────────────────────────────

def main():
    display_welcome()
    vocabulary = build_vocabulary(job_roles)

    while True:
        user_skills              = get_user_skills()
        recognized, unrecognized = check_recognized_skills(user_skills, vocabulary)

        # Use only recognized skills for matching; warn about the rest
        skills_to_match = recognized if recognized else user_skills

        print("\n   🔄 Running TF-IDF + Cosine Similarity pipeline...")
        results = get_recommendations(skills_to_match, job_roles, vocabulary, top_n=3)

        display_results(results, user_skills, unrecognized)

        again = input("   Try again with different skills? (yes / no): ").strip().lower()
        if again != "yes":
            print("\n   👋 Keep building. Keep learning. — DecodeLabs 🚀\n")
            break
        print("\n" + "-" * 58 + "\n")


if __name__ == "__main__":
    main()


   🤖  TECH STACK RECOMMENDER — DecodeLabs AI
   Project 3: AI Recommendation Logic | Batch 2026
   Enter your skills → get matched to the best job roles

📌 Example skills you can enter:
   python, sql, java, aws, docker, javascript,
   machine learning, html, css, linux, react,
   tensorflow, git, cloud, deep learning, figma,
   kotlin, data analysis, bash, networking...

   Type each skill and press Enter.
   Type 'done' when finished (minimum 3 skills).



   Skill #1:  sql


   ✅ Added: sql


   Skill #2:  aws


   ✅ Added: aws


...

   ✅ Added: python


   Skill #3:  sql


   ✅ Added: sql


   Skill #4:  done



   🔄 Running TF-IDF + Cosine Similarity pipeline...

   🎯  YOUR TOP JOB ROLE RECOMMENDATIONS

   Skills entered : machine learning, python, sql

   🥇  Rank 1: Data Scientist
         Score   : 60.11%  [████████████░░░░░░░░]
         Matched : machine learning, python, sql

   🥈  Rank 2: AI/ML Engineer
         Score   : 46.99%  [█████████░░░░░░░░░░░]
         Matched : machine learning, python

   🥉  Rank 3: Backend Developer
         Score   : 11.79%  [██░░░░░░░░░░░░░░░░░░]
         Matched : python, sql

